# AXAU15 + FH1223 — IPBus Control Notebook

Interfaccia di controllo per la scheda **AXAU15P** con FMC **FH1223**.

## Mappa registri IPBus (indirizzi word, 32 bit)

| Indirizzo | Nome | Accesso | Descrizione |
|-----------|------|---------|-------------|
| `0x00000000` | `ACQ_CTRL` | W | bit[0]=START, bit[1]=STOP |
| `0x00000001` | `ACQ_STATUS` | R | bit[0]=chan_up, bit[1]=acq_running |
| `0x00000002` | `ACQ_WORDS` | R | Parole acquisite (free-running) |
| `0x00000100` | `RC_DATA_LO` | W | Parola bassa 64-bit per Aurora TX |
| `0x00000101` | `RC_DATA_HI` | W | Parola alta 64-bit → push in FIFO |
| `0x00000102` | `RC_STATUS` | R | bit[0]=full, bit[1]=empty |
| `0x00001000+i`| `BRAM[i]` | R | Dati acquisiti (i=0..16383) |

## Dipendenze
```
pip install numpy matplotlib
```
L'IPBus transport è implementato direttamente con socket UDP (nessuna dipendenza da `uhal`).

In [ ]:
import socket
import struct
import time
import numpy as np
import matplotlib.pyplot as plt

# ── Configurazione board ──────────────────────────────────────────────────────
FPGA_IP   = '192.168.200.1'   # impostato in top_axau15.vhd
FPGA_PORT = 50001             # porta UDP IPBus standard
TIMEOUT   = 2.0               # secondi

# ── Indirizzi registri (word address, 32-bit) ─────────────────────────────────
ADDR_ACQ_CTRL   = 0x00000000
ADDR_ACQ_STATUS = 0x00000001
ADDR_ACQ_WORDS  = 0x00000002
ADDR_RC_LO      = 0x00000100
ADDR_RC_HI      = 0x00000101
ADDR_RC_STATUS  = 0x00000102
ADDR_BRAM_BASE  = 0x00001000
BRAM_DEPTH      = 16384       # 2^14 parole da 32 bit = 64 KB

print(f'Target: {FPGA_IP}:{FPGA_PORT}')

In [ ]:
# ── IPBus UDP transport (packet ID auto-incrementante) ───────────────────────

class IPBus:
    """IPBus v2.0 client minimale su UDP."""

    PROTO_VERSION = 0x20

    def __init__(self, host, port=50001, timeout=2.0):
        self.host    = host
        self.port    = port
        self._pkt_id = 1
        self._sock   = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self._sock.settimeout(timeout)
        self._sock.connect((host, port))

    # ── IPBus packet header ───────────────────────────────────────────────────
    def _header(self):
        # 0x200000f0 | (pkt_id << 8) | type=0 (control)
        hdr = (self.PROTO_VERSION << 24) | (self._pkt_id << 8) | 0xF0
        self._pkt_id = (self._pkt_id + 1) & 0xFFFF
        return hdr

    # ── Transaction words ─────────────────────────────────────────────────────
    @staticmethod
    def _read_tx(addr, nwords=1):
        # type=0 (read), words=nwords, id=0, info_code=0xF
        tx  = (0x0F << 28) | (nwords << 8) | 0x0F
        return [tx, addr]

    @staticmethod
    def _write_tx(addr, data):
        if isinstance(data, int):
            data = [data]
        nwords = len(data)
        tx = (0x1F << 28) | (nwords << 8) | 0x0F
        return [tx, addr] + list(data)

    # ── Send/receive helpers ──────────────────────────────────────────────────
    def _send(self, words):
        pkt = struct.pack(f'>{len(words)}I', *words)
        self._sock.send(pkt)
        resp = self._sock.recv(4096)
        return struct.unpack(f'>{len(resp)//4}I', resp)

    # ── Public API ────────────────────────────────────────────────────────────
    def read(self, addr):
        """Legge un singolo registro a 32 bit."""
        words  = [self._header()] + self._read_tx(addr, 1)
        resp   = self._send(words)
        # resp: [header, tx_hdr, data]
        return resp[2]

    def write(self, addr, value):
        """Scrive un valore 32 bit in un registro."""
        words = [self._header()] + self._write_tx(addr, [value])
        self._send(words)

    def read_block(self, addr, nwords):
        """Legge un blocco di nwords parole a partire da addr."""
        # Suddivide in chunk da 255 parole (limite IPBus per transazione)
        CHUNK = 255
        result = []
        for offset in range(0, nwords, CHUNK):
            n     = min(CHUNK, nwords - offset)
            words = [self._header()] + self._read_tx(addr + offset, n)
            resp  = self._send(words)
            result.extend(resp[2:2+n])
        return np.array(result, dtype=np.uint32)

    def close(self):
        self._sock.close()


# Crea il client
ipb = IPBus(FPGA_IP, FPGA_PORT, TIMEOUT)
print('IPBus client pronto.')

---
## 1. Verifica connessione — lettura STATUS

In [ ]:
status = ipb.read(ADDR_ACQ_STATUS)
chan_up     = bool(status & 0x1)
acq_running = bool((status >> 1) & 0x1)

print(f'STATUS         = 0x{status:08X}')
print(f'  chan_up      = {chan_up}  (Aurora link up)')
print(f'  acq_running  = {acq_running}')

---
## 2. Acquisizione dati

In [ ]:
def start_acquisition():
    """Avvia l'acquisizione (impulso START)."""
    ipb.write(ADDR_ACQ_CTRL, 0x1)   # bit0 = START
    ipb.write(ADDR_ACQ_CTRL, 0x0)   # torna a 0
    print('Acquisizione avviata.')


def stop_acquisition():
    """Ferma l'acquisizione (impulso STOP)."""
    ipb.write(ADDR_ACQ_CTRL, 0x2)   # bit1 = STOP
    ipb.write(ADDR_ACQ_CTRL, 0x0)
    words = ipb.read(ADDR_ACQ_WORDS)
    print(f'Acquisizione fermata. Parole scritte: {words}')
    return words


def wait_for_full(timeout_s=10.0):
    """Aspetta che il buffer sia pieno (acq_running=0 dopo start)."""
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        s = ipb.read(ADDR_ACQ_STATUS)
        if not (s >> 1) & 1:
            return True
        time.sleep(0.1)
    return False


# ── Esegui acquisizione ───────────────────────────────────────────────────────
start_acquisition()

# Attendi il completamento oppure ferma manualmente
print('In attesa di fine acquisizione (buffer pieno o STOP)...')
ok = wait_for_full(timeout_s=10.0)

if ok:
    nwords = ipb.read(ADDR_ACQ_WORDS)
    print(f'Completata automaticamente. Parole: {nwords}')
else:
    nwords = stop_acquisition()

---
## 3. Lettura BRAM e plot

In [ ]:
# Quante parole leggere (limita a quelle acquisite)
nread = min(int(nwords), BRAM_DEPTH)
print(f'Lettura di {nread} parole dalla BRAM...')

t0 = time.time()
data = ipb.read_block(ADDR_BRAM_BASE, nread)
dt  = time.time() - t0

print(f'Lette {len(data)} parole in {dt:.2f} s  ({len(data)*4/1024:.1f} kB)')
print(f'Primi 8 valori: {[hex(x) for x in data[:8]]}')

In [ ]:
# ── Plot dei dati acquisiti ───────────────────────────────────────────────────
samples = data.astype(np.int32)   # interpreta come int con segno

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Waveform completa
axes[0].plot(samples, linewidth=0.5)
axes[0].set_title('Dati acquisiti — BRAM completa')
axes[0].set_xlabel('Indice parola')
axes[0].set_ylabel('Valore (int32)')
axes[0].grid(True, alpha=0.3)

# Zoom sui primi 256 campioni
n_zoom = min(256, len(samples))
axes[1].plot(samples[:n_zoom], marker='.', markersize=3)
axes[1].set_title(f'Zoom — primi {n_zoom} campioni')
axes[1].set_xlabel('Indice parola')
axes[1].set_ylabel('Valore (int32)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Min: {samples.min():#010x}  Max: {samples.max():#010x}  Mean: {samples.mean():.1f}')

---
## 4. Remote control — invio configurazione via Aurora TX

In [ ]:
def send_config_word(hi, lo):
    """
    Invia una parola di configurazione a 64 bit al dispositivo remoto
    tramite Aurora TX.
    Scrivere HI (0x101) triggera il push in FIFO → Aurora.
    """
    # Controlla che la FIFO non sia piena
    fifo_st = ipb.read(ADDR_RC_STATUS)
    if fifo_st & 0x1:
        raise RuntimeError('FIFO remote_ctrl piena — aspetta prima di reinviare')
    ipb.write(ADDR_RC_LO, lo & 0xFFFFFFFF)
    ipb.write(ADDR_RC_HI, hi & 0xFFFFFFFF)  # push atomico


def rc_fifo_status():
    s = ipb.read(ADDR_RC_STATUS)
    return {'full': bool(s & 0x1), 'empty': bool((s >> 1) & 0x1)}


# Esempio: invia la parola 0xDEADBEEF_CAFEBABE
print('FIFO status prima:', rc_fifo_status())
send_config_word(hi=0xDEADBEEF, lo=0xCAFEBABE)
print('Parola inviata: HI=0xDEADBEEF  LO=0xCAFEBABE')
print('FIFO status dopo:', rc_fifo_status())

---
## 5. Utility — lettura rapida di un singolo indirizzo

In [ ]:
def dump_registers():
    """Stampa i registri principali in formato leggibile."""
    regs = {
        'ACQ_STATUS': ipb.read(ADDR_ACQ_STATUS),
        'ACQ_WORDS' : ipb.read(ADDR_ACQ_WORDS),
        'RC_STATUS' : ipb.read(ADDR_RC_STATUS),
    }
    print('─' * 40)
    for name, val in regs.items():
        print(f'  {name:<14} = 0x{val:08X}  ({val})')
    print('─' * 40)
    s = regs['ACQ_STATUS']
    print(f'  chan_up     = {bool(s & 1)}')
    print(f'  acq_running = {bool((s>>1) & 1)}')


dump_registers()

---
## 6. Salvataggio dati su file

In [ ]:
import pathlib, datetime

out_dir = pathlib.Path('../data')
out_dir.mkdir(exist_ok=True)

ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
fname = out_dir / f'acq_{ts}.npy'
np.save(fname, data)
print(f'Salvato: {fname}  ({fname.stat().st_size} byte)')

In [ ]:
# Chiudi la socket quando hai finito
ipb.close()
print('Connessione chiusa.')